<a href="https://colab.research.google.com/github/simonemedeiros/OIBSIP/blob/main/Mini_Autoresearch_Agentes_IA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Mini-Autoresearch: experimentação autônoma com um agente de IA

### Laboratório didático inspirado no `autoresearch` de Andrej Karpathy

Neste notebook, um **agente de linguagem local** escolhe uma modificação, um modelo pequeno é realmente treinado, uma métrica de validação é calculada e a mudança é **mantida (KEEP)** ou **descartada (DISCARD)**. O resultado de cada rodada entra no histórico usado para escolher a próxima.

> **Importante:** esta é uma **adaptação educacional**, não o runtime oficial do Autoresearch.  
> O projeto original modifica `train.py`, treina um LLM por um orçamento fixo de 5 minutos em uma GPU NVIDIA e otimiza `val_bpb`. Aqui usamos um modelo de classificação pequeno, um espaço de ações restrito e épocas fixas para que os experimentos caibam em poucos minutos de aula.

**Objetivo do laboratório:** observar o ciclo

`hipótese → modificação → treinamento → avaliação → KEEP/DISCARD → próxima hipótese`


## Roteiro

1. Rode a célula **1. Instalação**.
2. Rode a célula **2. Preparar o experimento**.
3. Rode a célula **3. Carregar o agente local** e espere aparecer `Agente pronto`.
4. Rode **4. Executar 3 experimentos**.
5. Observe, em cada rodada:
   - qual ação o agente escolheu;
   - qual configuração mudou;
   - qual foi a `validation loss`;
   - se a mudança foi `KEEP` ou `DISCARD`.

A célula do agente tenta usar um pequeno LLM local. Se o download do modelo falhar, o notebook ativa automaticamente um **modo de contingência** para que o laboratório continue e avisa claramente que esse modo não é um LLM.


# 1. Instalação

In [ ]:
%pip -q install "transformers>=4.44" "accelerate>=0.33"

print("Dependências instaladas.")


# 2. Preparar o experimento

In [ ]:
import copy
import json
import random
import re
import time
from dataclasses import dataclass

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from IPython.display import display

SEED = 42
TRAIN_DEVICE = torch.device("cpu")  # o modelo experimental é minúsculo; CPU é mais estável para a demonstração

def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything()

# ---------------------------------------------------------------------
# Dados: sklearn digits (1.797 imagens 8x8, 10 classes)
# Não exige download adicional.
# ---------------------------------------------------------------------
digits = load_digits()
X = digits.data.astype(np.float32)
y = digits.target.astype(np.int64)

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=SEED
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=SEED
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train).astype(np.float32)
X_val = scaler.transform(X_val).astype(np.float32)
X_test = scaler.transform(X_test).astype(np.float32)

def make_loader(X_array, y_array, batch_size, shuffle):
    ds = TensorDataset(torch.tensor(X_array), torch.tensor(y_array))
    generator = torch.Generator().manual_seed(SEED)
    return DataLoader(
        ds,
        batch_size=batch_size,
        shuffle=shuffle,
        generator=generator
    )

# ---------------------------------------------------------------------
# Modelo experimental: uma rede neural pequena.
# ---------------------------------------------------------------------
class SmallMLP(nn.Module):
    def __init__(self, hidden_dim=32, dropout=0.0):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(64, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 10),
        )

    def forward(self, x):
        return self.net(x)

def train_and_evaluate(config, seed=SEED, split="val"):
    """Treina do zero e devolve loss, accuracy, tempo e o modelo."""
    seed_everything(seed)

    model = SmallMLP(
        hidden_dim=config["hidden_dim"],
        dropout=config["dropout"],
    ).to(TRAIN_DEVICE)

    train_loader = make_loader(
        X_train, y_train,
        batch_size=config["batch_size"],
        shuffle=True,
    )

    if split == "val":
        eval_X, eval_y = X_val, y_val
    elif split == "test":
        eval_X, eval_y = X_test, y_test
    else:
        raise ValueError("split deve ser 'val' ou 'test'.")

    eval_loader = make_loader(eval_X, eval_y, batch_size=256, shuffle=False)

    if config["optimizer"] == "SGD":
        optimizer = torch.optim.SGD(
            model.parameters(),
            lr=config["lr"],
            momentum=0.9,
            weight_decay=config["weight_decay"],
        )
    elif config["optimizer"] == "Adam":
        optimizer = torch.optim.Adam(
            model.parameters(),
            lr=config["lr"],
            weight_decay=config["weight_decay"],
        )
    elif config["optimizer"] == "AdamW":
        optimizer = torch.optim.AdamW(
            model.parameters(),
            lr=config["lr"],
            weight_decay=config["weight_decay"],
        )
    else:
        raise ValueError(f"Optimizer desconhecido: {config['optimizer']}")

    criterion = nn.CrossEntropyLoss()
    start = time.time()

    model.train()
    for _ in range(config["epochs"]):
        for xb, yb in train_loader:
            xb = xb.to(TRAIN_DEVICE)
            yb = yb.to(TRAIN_DEVICE)

            optimizer.zero_grad(set_to_none=True)
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()

    elapsed = time.time() - start

    model.eval()
    total_loss = 0.0
    total = 0
    correct = 0

    with torch.no_grad():
        for xb, yb in eval_loader:
            xb = xb.to(TRAIN_DEVICE)
            yb = yb.to(TRAIN_DEVICE)
            logits = model(xb)
            loss = criterion(logits, yb)

            total_loss += loss.item() * yb.size(0)
            total += yb.size(0)
            correct += (logits.argmax(dim=1) == yb).sum().item()

    return {
        "loss": total_loss / total,
        "accuracy": correct / total,
        "seconds": elapsed,
        "model": model,
    }

# ---------------------------------------------------------------------
# Configuração inicial.
# Mantemos o número de épocas fixo para tornar a aula rápida e previsível.
# ---------------------------------------------------------------------
BASE_CONFIG = {
    "hidden_dim": 32,
    "dropout": 0.0,
    "optimizer": "SGD",
    "lr": 0.01,
    "weight_decay": 0.0,
    "batch_size": 64,
    "epochs": 8,
}

# ---------------------------------------------------------------------
# Espaço de ações.
# Nesta versão didática o agente não edita qualquer linha de Python:
# ele escolhe entre mudanças seguras e interpretáveis.
# ---------------------------------------------------------------------
def A1(cfg):
    new = copy.deepcopy(cfg)
    new["lr"] = min(new["lr"] * 3.0, 0.10)
    return new, f"learning rate: {cfg['lr']:.5f} → {new['lr']:.5f}"

def A2(cfg):
    new = copy.deepcopy(cfg)
    new["lr"] = max(new["lr"] / 3.0, 0.0001)
    return new, f"learning rate: {cfg['lr']:.5f} → {new['lr']:.5f}"

def A3(cfg):
    new = copy.deepcopy(cfg)
    new["hidden_dim"] = min(new["hidden_dim"] * 2, 128)
    return new, f"hidden units: {cfg['hidden_dim']} → {new['hidden_dim']}"

def A4(cfg):
    new = copy.deepcopy(cfg)
    new["hidden_dim"] = max(new["hidden_dim"] // 2, 16)
    return new, f"hidden units: {cfg['hidden_dim']} → {new['hidden_dim']}"

def A5(cfg):
    new = copy.deepcopy(cfg)
    new["dropout"] = min(round(new["dropout"] + 0.2, 1), 0.4)
    return new, f"dropout: {cfg['dropout']:.1f} → {new['dropout']:.1f}"

def A6(cfg):
    new = copy.deepcopy(cfg)
    new["batch_size"] = max(new["batch_size"] // 2, 32)
    return new, f"batch size: {cfg['batch_size']} → {new['batch_size']}"

def A7(cfg):
    new = copy.deepcopy(cfg)
    new["batch_size"] = min(new["batch_size"] * 2, 128)
    return new, f"batch size: {cfg['batch_size']} → {new['batch_size']}"

def A8(cfg):
    new = copy.deepcopy(cfg)
    new["optimizer"] = "Adam"
    new["lr"] = 0.003
    return new, f"optimizer: {cfg['optimizer']} → Adam; learning rate → 0.003"

def A9(cfg):
    new = copy.deepcopy(cfg)
    new["optimizer"] = "AdamW"
    new["lr"] = 0.003
    new["weight_decay"] = 0.0001
    return new, f"optimizer: {cfg['optimizer']} → AdamW; lr → 0.003; weight decay → 1e-4"

def A10(cfg):
    new = copy.deepcopy(cfg)
    new["weight_decay"] = 0.001
    return new, f"weight decay: {cfg['weight_decay']:.4g} → 0.001"

ACTIONS = {
    "A1": {
        "label": "Increase learning rate ×3",
        "description": "Try faster optimization by multiplying the current learning rate by 3.",
        "apply": A1,
        "valid": lambda c: c["lr"] < 0.10,
    },
    "A2": {
        "label": "Decrease learning rate ÷3",
        "description": "Try a more conservative optimization step.",
        "apply": A2,
        "valid": lambda c: c["lr"] > 0.0001,
    },
    "A3": {
        "label": "Widen the hidden layer ×2",
        "description": "Increase model capacity by doubling hidden units, up to 128.",
        "apply": A3,
        "valid": lambda c: c["hidden_dim"] < 128,
    },
    "A4": {
        "label": "Narrow the hidden layer ÷2",
        "description": "Reduce model capacity by halving hidden units, down to 16.",
        "apply": A4,
        "valid": lambda c: c["hidden_dim"] > 16,
    },
    "A5": {
        "label": "Add +0.2 dropout",
        "description": "Increase regularization by adding dropout, up to 0.4.",
        "apply": A5,
        "valid": lambda c: c["dropout"] < 0.4,
    },
    "A6": {
        "label": "Use a smaller batch",
        "description": "Halve the batch size, down to 32.",
        "apply": A6,
        "valid": lambda c: c["batch_size"] > 32,
    },
    "A7": {
        "label": "Use a larger batch",
        "description": "Double the batch size, up to 128.",
        "apply": A7,
        "valid": lambda c: c["batch_size"] < 128,
    },
    "A8": {
        "label": "Switch optimizer to Adam",
        "description": "Switch to Adam and use learning rate 0.003.",
        "apply": A8,
        "valid": lambda c: c["optimizer"] != "Adam",
    },
    "A9": {
        "label": "Switch optimizer to AdamW",
        "description": "Switch to AdamW with lr 0.003 and weight decay 1e-4.",
        "apply": A9,
        "valid": lambda c: c["optimizer"] != "AdamW",
    },
    "A10": {
        "label": "Add weight decay",
        "description": "Set weight decay to 0.001.",
        "apply": A10,
        "valid": lambda c: c["weight_decay"] < 0.001,
    },
}

print(f"Dados prontos: {len(X_train)} treino / {len(X_val)} validação / {len(X_test)} teste")
print("Modelo experimental e espaço de ações prontos.")


# 3. Carregar o agente local

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

# Modelo pequeno o suficiente para a aula (~135M parâmetros; arquivo principal ~269 MB).
AGENT_MODEL_NAME = "HuggingFaceTB/SmolLM2-135M-Instruct"

LLM_READY = False
agent_tokenizer = None
agent_model = None
AGENT_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

try:
    print(f"Carregando {AGENT_MODEL_NAME} em {AGENT_DEVICE}...")
    agent_tokenizer = AutoTokenizer.from_pretrained(AGENT_MODEL_NAME)

    dtype = torch.float16 if AGENT_DEVICE == "cuda" else torch.float32
    agent_model = AutoModelForCausalLM.from_pretrained(
        AGENT_MODEL_NAME,
        torch_dtype=dtype,
        low_cpu_mem_usage=True,
    ).to(AGENT_DEVICE)

    agent_model.eval()
    LLM_READY = True
    print("Agente pronto: LLM local.")
except Exception as e:
    print("Não foi possível carregar o LLM local.")
    print("O laboratório continuará em MODO DE CONTINGÊNCIA (não-LLM).")
    print("Detalhe técnico:", type(e).__name__, str(e)[:220])


# 4. Lógica do agente e do ciclo experimental

In [ ]:
def compact_config(cfg):
    return (
        f"hidden={cfg['hidden_dim']}, dropout={cfg['dropout']}, "
        f"optimizer={cfg['optimizer']}, lr={cfg['lr']:.5f}, "
        f"weight_decay={cfg['weight_decay']:.4g}, batch={cfg['batch_size']}, "
        f"epochs={cfg['epochs']}"
    )

def available_action_ids(current_config, tried_action_ids):
    ids = []
    for action_id, spec in ACTIONS.items():
        if action_id not in tried_action_ids and spec["valid"](current_config):
            ids.append(action_id)
    return ids

def history_for_prompt(history):
    lines = []
    for row in history:
        if row["experiment"] == 0:
            lines.append(
                f"- Baseline: val_loss={row['val_loss']:.4f}, "
                f"val_accuracy={row['val_accuracy']:.3f}"
            )
        else:
            lines.append(
                f"- Experiment {row['experiment']}: {row['action_id']} "
                f"({row['status']}), val_loss={row['val_loss']:.4f}, "
                f"best_after={row['best_after']:.4f}"
            )
    return "\n".join(lines)

def ask_local_llm(current_config, history, tried_action_ids):
    available = available_action_ids(current_config, tried_action_ids)
    if not available:
        raise RuntimeError("Não há ações disponíveis.")

    actions_text = "\n".join(
        f"{aid}: {ACTIONS[aid]['label']} — {ACTIONS[aid]['description']}"
        for aid in available
    )

    prompt = f"""
You are a cautious machine-learning research agent.

GOAL:
Minimize validation cross-entropy loss. Lower is better.

CURRENT BEST CONFIGURATION:
{compact_config(current_config)}

EXPERIMENT HISTORY:
{history_for_prompt(history)}

AVAILABLE ACTIONS:
{actions_text}

Choose exactly ONE available action for the next experiment.
Use the previous experimental evidence when it is relevant.
Do not invent a new action.
Prefer a simple, interpretable experiment.

Return exactly two lines:
ACTION=<one available action id>
REASON=<one short sentence>
""".strip()

    messages = [
        {
            "role": "system",
            "content": "You select the next ML experiment from a constrained action list."
        },
        {"role": "user", "content": prompt},
    ]

    text = agent_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = agent_tokenizer(text, return_tensors="pt").to(AGENT_DEVICE)

    with torch.no_grad():
        outputs = agent_model.generate(
            **inputs,
            max_new_tokens=70,
            do_sample=False,
            pad_token_id=agent_tokenizer.eos_token_id,
        )

    generated = outputs[0][inputs["input_ids"].shape[-1]:]
    response = agent_tokenizer.decode(generated, skip_special_tokens=True).strip()

    match = re.search(r"ACTION\s*=\s*(A\d+)", response, flags=re.I)
    reason_match = re.search(r"REASON\s*=\s*(.+)", response, flags=re.I | re.S)

    if match and match.group(1).upper() in available:
        action_id = match.group(1).upper()
    else:
        # Segundo parser: procura qualquer ID disponível no texto.
        action_id = None
        for aid in available:
            if re.search(rf"\b{re.escape(aid)}\b", response, flags=re.I):
                action_id = aid
                break

    reason = (
        reason_match.group(1).strip().splitlines()[0]
        if reason_match
        else response.splitlines()[0] if response else "No rationale returned."
    )

    if action_id is None:
        # Fallback de parsing, ainda dentro do modo LLM:
        # seleciona a primeira ação válida para que a aula não trave.
        action_id = available[0]
        reason = "The LLM response could not be parsed reliably; using a safe valid action."

    return action_id, reason, response

def contingency_agent(current_config, history, tried_action_ids):
    """
    Plano B para falha de download/inferência.
    Não é um LLM. A ordem foi escolhida para produzir um ciclo didático útil.
    """
    available = available_action_ids(current_config, tried_action_ids)

    # Prioridades dependem do estado atual e do histórico;
    # o objetivo aqui é manter o laboratório funcional, não simular um LLM.
    priority = ["A8", "A5", "A6", "A3", "A1", "A10", "A9", "A2", "A7", "A4"]
    for aid in priority:
        if aid in available:
            return (
                aid,
                "Contingency policy selected this valid experiment.",
                "[contingency mode: no LLM response]"
            )

    return available[0], "Contingency policy selected the remaining valid action.", ""

def propose_experiment(current_config, history, tried_action_ids):
    if LLM_READY:
        return ask_local_llm(current_config, history, tried_action_ids)
    return contingency_agent(current_config, history, tried_action_ids)

def evaluate_saved_model(model, split="test"):
    if split == "test":
        X_eval, y_eval = X_test, y_test
    else:
        X_eval, y_eval = X_val, y_val

    loader = make_loader(X_eval, y_eval, batch_size=256, shuffle=False)
    criterion = nn.CrossEntropyLoss()

    model.eval()
    total_loss = 0.0
    total = 0
    correct = 0

    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(TRAIN_DEVICE)
            yb = yb.to(TRAIN_DEVICE)
            logits = model(xb)
            loss = criterion(logits, yb)
            total_loss += loss.item() * yb.size(0)
            total += yb.size(0)
            correct += (logits.argmax(dim=1) == yb).sum().item()

    return total_loss / total, correct / total

def run_autoresearch(n_experiments=3):
    if not isinstance(n_experiments, int):
        raise TypeError("n_experiments deve ser inteiro.")
    n_experiments = max(1, min(n_experiments, 10))

    mode = "LLM local" if LLM_READY else "CONTINGÊNCIA (não-LLM)"
    print("=" * 72)
    print("MINI-AUTORESEARCH")
    print(f"Agente: {mode}")
    print("Objetivo: minimizar validation cross-entropy loss")
    print("=" * 72)

    current_config = copy.deepcopy(BASE_CONFIG)
    tried_action_ids = set()
    history = []

    print("\n[BASELINE] Treinando configuração inicial...")
    baseline = train_and_evaluate(current_config)
    best_loss = baseline["loss"]
    best_accuracy = baseline["accuracy"]
    best_model = baseline["model"]

    history.append({
        "experiment": 0,
        "action_id": "BASE",
        "action": "Baseline",
        "change": "—",
        "val_loss": best_loss,
        "val_accuracy": best_accuracy,
        "status": "KEEP",
        "best_after": best_loss,
        "seconds": baseline["seconds"],
        "agent_reason": "Initial baseline.",
    })

    print(f"Validation loss: {best_loss:.4f}")
    print(f"Validation accuracy: {best_accuracy:.3f}")
    print("Status: KEEP")

    for exp in range(1, n_experiments + 1):
        print("\n" + "─" * 72)
        print(f"EXPERIMENTO {exp}/{n_experiments}")
        print("─" * 72)

        action_id, reason, raw_response = propose_experiment(
            current_config=current_config,
            history=history,
            tried_action_ids=tried_action_ids,
        )
        tried_action_ids.add(action_id)

        spec = ACTIONS[action_id]
        candidate_config, change = spec["apply"](current_config)

        # Proteção contra ação que, por algum motivo, não alterou o estado.
        if candidate_config == current_config:
            remaining = available_action_ids(current_config, tried_action_ids)
            if remaining:
                action_id = remaining[0]
                tried_action_ids.add(action_id)
                spec = ACTIONS[action_id]
                candidate_config, change = spec["apply"](current_config)
                reason = "Previous action produced no state change; selected another valid action."

        print(f"Hipótese/ação: {action_id} — {spec['label']}")
        print(f"Justificativa do agente: {reason}")
        print(f"Modificação: {change}")
        print("Treinando...")

        candidate = train_and_evaluate(candidate_config)
        candidate_loss = candidate["loss"]
        candidate_accuracy = candidate["accuracy"]

        improved = candidate_loss < (best_loss - 1e-4)

        if improved:
            status = "KEEP"
            current_config = candidate_config
            best_loss = candidate_loss
            best_accuracy = candidate_accuracy
            best_model = candidate["model"]
        else:
            status = "DISCARD"

        history.append({
            "experiment": exp,
            "action_id": action_id,
            "action": spec["label"],
            "change": change,
            "val_loss": candidate_loss,
            "val_accuracy": candidate_accuracy,
            "status": status,
            "best_after": best_loss,
            "seconds": candidate["seconds"],
            "agent_reason": reason,
        })

        print(f"Validation loss: {candidate_loss:.4f}  (menor é melhor)")
        print(f"Validation accuracy: {candidate_accuracy:.3f}")
        print(f"→ {status}")
        print(f"Melhor loss após esta rodada: {best_loss:.4f}")

    df = pd.DataFrame(history)

    test_loss, test_accuracy = evaluate_saved_model(best_model, split="test")

    print("\n" + "=" * 72)
    print("RESULTADO FINAL")
    print("=" * 72)
    print("Melhor configuração:")
    print(compact_config(current_config))
    print(f"\nMelhor validation loss: {best_loss:.4f}")
    print(f"Teste final (não usado pelo agente): loss={test_loss:.4f}, accuracy={test_accuracy:.3f}")

    print("\nHistórico:")
    display(
        df[[
            "experiment", "action_id", "action", "val_loss",
            "val_accuracy", "status", "best_after"
        ]].round(4)
    )

    # Visualização: cada ponto é a métrica produzida pelo experimento.
    fig, ax = plt.subplots(figsize=(8, 4.5))
    ax.plot(df["experiment"], df["val_loss"], marker="o", label="candidate val_loss")
    ax.plot(df["experiment"], df["best_after"], marker="s", label="best-so-far")
    ax.set_xlabel("Experimento")
    ax.set_ylabel("Validation loss (↓)")
    ax.set_title("Histórico experimental")
    ax.set_xticks(df["experiment"])
    ax.legend()
    ax.grid(alpha=0.25)
    plt.show()

    return {
        "history": df,
        "best_config": current_config,
        "best_val_loss": best_loss,
        "best_val_accuracy": best_accuracy,
        "test_loss": test_loss,
        "test_accuracy": test_accuracy,
        "agent_mode": mode,
    }

print("Ciclo experimental pronto.")


# 5. Executar experimentos

In [ ]:
resultado = run_autoresearch(n_experiments=8)


## O que observar

Enquanto o loop roda, tente identificar:

- **Hipótese / decisão:** qual modificação o agente escolheu?
- **Ação:** o que mudou na configuração?
- **Evidência:** a `validation loss` realmente melhorou?
- **Decisão:** foi `KEEP` ou `DISCARD`?
- **Memória:** o resultado anterior aparece no histórico usado na rodada seguinte?

### Pergunta para discussão

> O agente está “fazendo pesquisa” ou apenas automatizando uma busca experimental?

Não existe uma única resposta correta. Compare o que você acabou de executar com o Autoresearch original: lá o agente pode editar livremente `train.py`; aqui restringimos deliberadamente o espaço de ações para tornar a atividade rápida, segura e reproduzível.



## O que esta adaptação preserva — e o que muda

| Elemento | Autoresearch oficial | Este laboratório |
|---|---|---|
| Quem escolhe o próximo experimento? | Coding agent | Pequeno LLM local |
| O que pode mudar? | Código em `train.py` | Conjunto restrito de mudanças |
| Modelo treinado | Pequeno LLM | MLP para classificação de dígitos |
| Métrica principal | `val_bpb` ↓ | validation cross-entropy ↓ |
| Orçamento | 5 min de wall-clock por experimento | 8 épocas por experimento |
| Hardware | GPU NVIDIA | CPU; GPU opcional para o agente |
| KEEP / DISCARD | Sim | Sim |
| Resultado alimenta a próxima rodada | Sim | Sim |

Essa diferença é proposital: o objetivo aqui é **observar o mecanismo**, não reproduzir a escala computacional do projeto original.



## Fontes

- Andrej Karpathy, **autoresearch** — repositório oficial:  
  https://github.com/karpathy/autoresearch

- `program.md` do Autoresearch — regras do loop experimental:  
  https://github.com/karpathy/autoresearch/blob/master/program.md

- Hugging Face, **SmolLM2-135M-Instruct** — modelo local usado como agente:  
  https://huggingface.co/HuggingFaceTB/SmolLM2-135M-Instruct

- Dataset `digits`, scikit-learn:  
  https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_digits.html
